### New experiments using OSM. 
* We will pull out the relevant data in tags of osm
* Then we will filter to Exeter

In [ ]:
import pyrosm
import geopandas as gpd
import pandas as pd

PBF_FILE = r"osm_data\devon-260326.osm.pbf"

print("Reading OSM data...")
osm = pyrosm.OSM(PBF_FILE)

# Helper function to safely save a layer
def save_layer(gdf, name):
    if gdf is not None and len(gdf) > 0:
        print(f"{name}: {len(gdf):,} features")
        gdf.to_file(f"devon_{name}.gpkg", driver="GPKG")
        print(f"✓ Saved devon_{name}.gpkg")
    else:
        print(f"{name}: empty or not found")

# ── Buildings ─────────────────────────────────────────────
print("\nReading buildings...")
buildings = osm.get_buildings()
save_layer(buildings, "buildings")

# ── Network / Roads ───────────────────────────────────────
print("\nReading roads/network...")
network = osm.get_network(network_type="all")
save_layer(network, "network")

# ── Landuse ───────────────────────────────────────────────
print("\nReading landuse...")
landuse = osm.get_landuse()
save_layer(landuse, "landuse")

# ── POIs ──────────────────────────────────────────────────
print("\nReading POIs...")
pois = osm.get_pois()
save_layer(pois, "pois")

# ── Natural features ──────────────────────────────────────
print("\nReading natural features...")
natural = osm.get_natural()
save_layer(natural, "natural")

# ── Waterways ─────────────────────────────────────────────
print("\nReading waterways...")
waterways = osm.get_data_by_custom_criteria(custom_filter={"waterway": True})
save_layer(waterways, "waterways")

# ── Boundaries ────────────────────────────────────────────
print("\nReading boundaries...")
boundaries = osm.get_data_by_custom_criteria(custom_filter={"boundary": True})
save_layer(boundaries, "boundaries")

print("\nDone!")

### Post filtering of OSM data 
* Removing irrelevant columns
* Finding filter columns

#### Buildings

In [ ]:
import geopandas as gpd
gdf_polygon = gpd.read_file(r"osm_data\devon_buildings.gpkg")
gdf_polygon.columns

In [ ]:
recommended_columns = ['addr:city', 'addr:country', 'addr:housenumber', 'addr:housename',
       'addr:postcode', 'addr:place', 'addr:street','name',
       'opening_hours', 'website', 'building', 'amenity', 'building:flats', 'building:levels',
       'building:material', 'building:min_level', 'building:use','craft',
       'height', 'internet_access', 'landuse', 'levels', 'office', 'shop',
       'source','geometry']

In [ ]:
gdf_polygon.building.isna().any()


In [ ]:
invalid = gdf_polygon[~gdf_polygon.is_valid]
print(len(invalid))

In [ ]:
gdf_polygon.building.unique()

#### Landuse

In [ ]:
gdf_landuse = gpd.read_file(r"osm_data\devon_landuse.gpkg")
gdf_landuse.columns

In [ ]:
invalid = gdf_landuse[~gdf_landuse.is_valid]
print(len(invalid))

In [ ]:
gdf_landuse.crs

In [ ]:
gdf_landuse.head()

In [ ]:
gdf_landuse.landuse.unique()

In [ ]:
gdf_landuse.landuse.isna().any()

#### Natural

In [ ]:
gdf_natural = gpd.read_file(r"osm_data\devon_natural.gpkg")
gdf_natural.columns

In [ ]:
invalid = gdf_natural[~gdf_natural.is_valid]
print(len(invalid))

In [ ]:
gdf_natural.crs

In [ ]:
gdf_natural.natural.unique()

#### POI

In [ ]:
gdf_pois = gpd.read_file(r"osm_data\devon_pois.gpkg")
gdf_pois.columns

In [ ]:
invalid = gdf_pois[~gdf_pois.is_valid]
print(len(invalid))

In [ ]:
gdf_pois.crs

#### waterways

In [ ]:
gdf_waterways = gpd.read_file(r"osm_data\devon_waterways.gpkg")
gdf_waterways.columns

In [ ]:
invalid = gdf_waterways[~gdf_waterways.is_valid]
print(len(invalid))

In [ ]:
gdf_waterways.crs

In [ ]:
gdf_waterways.waterway.unique()

### Boundaries

In [ ]:
gdf_boundaries = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_boundaries.columns

In [ ]:
invalid = gdf_boundaries[~gdf_boundaries.is_valid]
print(len(invalid))

In [ ]:
gdf_boundaries.crs

In [ ]:
gdf_boundaries.name.unique()

### Finally converting all to EPSG 27700

In [ ]:
import os
import geopandas as gpd
for filename in os.listdir(r"osm_data"):
    if filename.endswith(".gpkg"):
        gdf = gpd.read_file(os.path.join(r"osm_data", filename))
        gdf = gdf.to_crs(epsg=27700)
        gdf = gdf[gdf.is_valid]
        gdf.to_file(os.path.join(r"osm_data", filename), driver="GPKG")

In [ ]:
import geopandas as gpd
gdf_admin = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_admin.name.value_counts()

In [ ]:
print(gdf_admin.name.value_counts())

In [ ]:
gdf_admin[gdf_admin.name == "Dorset"]

In [ ]:
import joblib
data = joblib.load(r"artifacts\waterway_river_clyst_exeter_search.pkl").data
data.columns

In [ ]:
data

### DOCUMENTATION : How to use this tool

In [17]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import joblib
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"log",diff_dir=None).initialize_all_agents()

human_send_message(message="From our conversation the first output showed 6 restaruants and the second showed more than 6 why? Return the final artifact with the data and a plot",target_agent=[agent_archiecture["host_agent"]])

Openai and ngd key set successfully
Model chosen gpt-4.1


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 22158: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\traitlets\config\application.

Model chosen gpt-4.1


'Here’s a clear explanation for the difference you observed, based on the available artifacts and data:\n\nReason for the Difference:\n- The first output (6 restaurants) was likely a summary or a sample, not the full filtered dataset.\n- The actual filtered dataset for restaurants between 1km and 2km from the University of Exeter is stored in the artifact filtered_exeter_restaurants, which contains 88 restaurants.\n- The second output (with the plot and circles) used the full filtered dataset, hence showing more than 6 restaurants.\n\nSummary:\n- The initial list was a small sample for illustration.\n- The full result, as shown in the plot and data artifact, includes all 88 restaurants that meet the 1–2km distance criterion.\n\nFinal Artifact:\n- The artifact with the complete data and plot is: restaurants_and_university_with_radii\n- Description: Folium map showing filtered restaurants (1–2km from university) and university locations with 1km and 2km radius circles.\n\nIf you want the

In [23]:
import shutil
import os
shutil.rmtree("artifacts")
shutil.rmtree("message_store")
os.makedirs("artifacts", exist_ok=True)
os.makedirs("message_store", exist_ok=True)

### Experiments

In [25]:
import pandas as pd

questions = pd.read_csv(r"evaluation\valid.csv")
questions

,Unnamed: 0,index,id,query,sql,query_type,expectation,dataset,attribute,notes
0,0,0,1,Show me everything OS knows about {input_locat...,"(SELECT osid, theme, description, geometry_wgs...",spatial_range,expect_sql,NaN,NaN,Need to combine all available layers.
1,1,1,2,Show me everything OS knows about 'electricity...,"(SELECT osid, theme, description, geometry_wgs...",attribute_filter,expect_sql,NaN,NaN,NaN
2,2,2,3,What information do you have on roofs for {inp...,"SELECT osid, theme, roofmaterial_primarymateri...",attribute_filter,expect_sql,building,"roofmaterial_primarymaterial, roofmaterial_sol...",NaN
3,3,3,4,What information do you have on wetlands in {i...,"SELECT osid, theme, description, geometry_wgs8...",attribute_filter,expect_sql,land,description,NaN
4,4,4,5,Show me all the buildings in {input_location}.,"SELECT osid, description, geometry_wgs84 FROM ...",spatial_range,expect_sql,building,NaN,NaN
5,5,5,6,Show me all the buildings that are within 1km ...,WITH location_buffer AS (SELECT st_union_agg(s...,proximity,expect_sql,building,NaN,NaN
6,6,6,7,Show me all commercial buildings that are with...,WITH location_buffer AS (SELECT st_union_agg(s...,"proximity,attribute_filter",expect_sql,building,buildinguse_addresscount_commercial,NaN
7,7,7,8,Show me all the buildings that are within 1km ...,WITH location_geometry AS (SELECT st_buffer(st...,proximity,expect_sql,building,NaN,NaN
8,8,8,9,Show me all the buildings that are near {input...,"SELECT osid, description, geometry_wgs84 FROM ...",spatial_range,expect_sql,building,NaN,NaN
9,9,9,10,Show me buildings that are between 1km and 2km...,WITH location_geometry AS (SELECT geometry FRO...,proximity,expect_sql,building,NaN,NaN


In [1]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import joblib
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"log",diff_dir=None).initialize_all_agents()

human_send_message(message="Find buildings in Topsham. Return the final artifact with the data and a plot",target_agent=[agent_archiecture["host_agent"]])
shutil.rmtree("artifacts")
shutil.rmtree("message_store")
os.makedirs("artifacts", exist_ok=True)
os.makedirs("message_store", exist_ok=True)

Openai and ngd key set successfully
Openai and ngd key set successfully
Model chosen o4-mini
Model chosen o4-mini
Model chosen gpt-4o-mini
Model chosen o4-mini
Model chosen gpt-4o
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4o
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4o
Model chosen o4-mini
Messages for human to process: I attempted to locate the named area 'Topsham' in the OSM boundary data for the UK, but no exact matches were found. Could you please provide a more precise name or context (e.g., 'Topsham, Devon' or a relation ID) so I can locate the correct boundary?
OFFLINE MODE : Please provide your response to the following query:
Model chosen o4-mini
